In [4]:
# Check GPU availability
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
# Install dependencies
# !pip install inference-gpu trackers==2.2.0 pycuda supervision

In [ ]:
# Import Libraries

from google.colab import files
from IPython.display import Video
import os
import numpy as np
from inference import get_model
from trackers import ByteTrackTracker, MotionEstimator, MotionAwareTraceAnnotator
import supervision as sv

In [ ]:
# Upload Video

print("Upload your video file (.mp4):")
uploaded = files.upload()

# Extract the filename dynamically
INPUT_VIDEO_NAME = list(uploaded.keys())[0]
print(f"Successfully uploaded: {INPUT_VIDEO_NAME}")

# Display the original uploaded video
# Note: The video must be in a web-compatible codec (like H.264) to display natively here.
Video(INPUT_VIDEO_NAME, embed=True, width=1080)

In [ ]:
# Object Definition

# Track 'Person' (0), 'Car' (2), 'Bird' (14)
TARGET_CLASS_IDS = [0, 2, 14]

CLASS_NAMES_DICT = {
    0: "Person",
    2: "Car",
    14: "Bird"
}

In [ ]:
# Control Parameters

# Minimum Confidence Threshold for every oject
CONFIDENCE_THRESHOLD = 0.2

# Maximum Non-Maximum Suppression - If more than 30% overlap then the lower NMS is dropped
NMS_THRESHOLD = 0.3

In [ ]:
# Motion Compensated Tracking Pipeline

# Set target paths based on the uploaded filename
TARGET_VIDEO_PATH = f"tracked_output_{INPUT_VIDEO_NAME}"

# Load Model and Tracker
model = get_model("rfdetr-large") # Roboflow DEtection TRansformer
tracker = ByteTrackTracker(minimum_consecutive_frames=3)

# Initialize the Motion Estimator for camera stabilization
motion_estimator = MotionEstimator(
    max_points=500,
    min_distance=10,
    quality_level=0.001,
    ransac_reproj_threshold=1.0,
)

# Visual Formatting

# Custom Color Palette
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# Box and Label Annotators
box_annotator = sv.BoxAnnotator(color=color, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(
    color=color, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK, text_scale=0.8
)

# Motion-Aware Trace Annotator
motion_aware_trace_annotator = MotionAwareTraceAnnotator(
    color=color, color_lookup=sv.ColorLookup.TRACK, thickness=2, trace_length=100
)

# Frame Processing Callback
def callback(frame, i):
    # 1. Estimate camera motion
    coord_transform = motion_estimator.update(frame)

    # 2. Detect objects
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)

    # 3. Filter for specific classes
    mask = np.isin(detections.class_id, TARGET_CLASS_IDS)
    detections = detections[mask]

    # 4. Update tracker
    detections = tracker.update(detections)

    # 5. Generate Custom Labels
    # This creates a list like: ["Person #1", "Car #2", "Person #3"]
    custom_labels = [
        f"{CLASS_NAMES_DICT.get(class_id, 'Unknown')} #{tracker_id}"
        for class_id, tracker_id
        in zip(detections.class_id, detections.tracker_id)
    ]

    # 6. Annotate frame
    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)
    annotated_image = motion_aware_trace_annotator.annotate(
        scene=annotated_image, detections=detections, custom_color_lookup=np.array(detections.tracker_id), coord_transform=coord_transform
    )

    # Pass our custom_labels list to the label_annotator
    annotated_image = label_annotator.annotate(
        scene=annotated_image, detections=detections, labels=custom_labels
    )

    return annotated_image

# Reset states before processing
tracker.reset()
motion_estimator.reset()
motion_aware_trace_annotator.reset()

print("Processing video with Motion Compensation... This may take a moment.")

# Process Video
sv.process_video(
    source_path=INPUT_VIDEO_NAME,
    target_path=TARGET_VIDEO_PATH,
    callback=callback,
    show_progress=True,
)

In [ ]:
# Display Object Tracked Video

# Output path for the compressed web-ready video
COMPRESSED_OUTPUT_PATH = f"compressed_tracked_{INPUT_VIDEO_NAME}"

print("Compressing video for browser display")

# Compress using ffmpeg
!ffmpeg -y -loglevel error -i "{TARGET_VIDEO_PATH}" -vcodec libx264 -crf 28 "{COMPRESSED_OUTPUT_PATH}"

print("Done! Displaying result:")

# Display the final tracked video
Video(COMPRESSED_OUTPUT_PATH, embed=True, width=1080)